<a href="https://colab.research.google.com/github/JuanZapa7a/AINavalEngineering/blob/main/NB14_Sequence_Models_RNN_LSTM_Real_Fuel_Forecasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **NB14 · Class 14 — Sequence Models: RNN, LSTM, and Real Fuel Forecasting**

## Block 3: AI — Deep Learning (continued)

`NB11`–`NB13` handled tabular data and images. This class covers the third major data shape: **sequences**, where a value's meaning depends on what came before it. We build the theory from the ground up — the RNN recurrence relation, the vanishing gradient problem, and LSTM's fix for it — then apply it for real: reframing the ship fuel-consumption data from `NB07`/`NB09` as what it actually is, a **time series per ship**, and training an LSTM to forecast next month's fuel consumption from the months before it.

### Learning objectives

By the end of this class, students will be able to:
- Explain why sequence data needs a different architecture than an MLP or a CNN.
- Explain the RNN recurrence relation and compute a small RNN's hidden states by hand.
- Explain the vanishing gradient problem and how LSTM's gated cell state addresses it.
- Reshape real tabular data with a natural time axis into proper sequences for a recurrent model, without leaking information across ships.
- Train and evaluate a real PyTorch LSTM, judged against a simple, honest baseline.

### Agenda (2-hour class)

| # | Class segment | Approx. duration | Type |
|---|---------------------|:---:|:---:|
| 1 | Recap, today's roadmap | 5 min | Theory |
| 2 | Why sequence models? Where MLPs and CNNs fall short | 10 min | Theory |
| 3 | The RNN recurrence relation, unrolled through time | 15 min | Theory + Practice |
| 4 | Worked example: computing an RNN's hidden states by hand | 15 min | Theory + Practice |
| 5 | The vanishing gradient problem | 10 min | Theory + Practice |
| 6 | LSTM: gates and cell state | 15 min | Theory |
| 7 | Real dataset: ship fuel data as a proper time series | 10 min | Practice |
| 8 | Hands-on: building sequences and training an LSTM forecaster | 20 min | Practice |
| 9 | Evaluating the LSTM against an honest baseline | 15 min | Practice |
| 10 | Summary, homework, next class | 5 min | Theory |

> Timings are approximate guidance, not a strict script — there are no scheduled breaks. If we cover everything with time to spare, class ends early; that can happen and is fine.


---

## 1. Recap: where we are

- **`NB11`**: perceptron → MLP, a first real PyTorch classifier.
- **`NB12`**: training deep networks properly — validation monitoring, dropout, early stopping, optimizers.
- **`NB13`**: Convolutional Neural Networks on real underwater hull images.
- **`NB14`** (today): sequence models — RNN, LSTM — applied to a real naval time series.

---

## 2. Why sequence models? Where MLPs and CNNs fall short

`NB11`'s MLP treated its inputs as an unordered bag of numbers. `NB13`'s CNN respects *spatial* structure, but not *temporal order*. Some real naval/ocean data is fundamentally **sequential**: a value's meaning depends on what came before it.

Look back at `ship_fuel_efficiency.csv` from `NB07`/`NB09`: each of the 120 ships has **12 monthly records**, January through December. Every model we've trained on it so far — Logistic Regression, Random Forest, K-Means — treated those 1,440 rows as independent, interchangeable examples, exactly like `NB07`'s classification/regression setup and `NB09`'s clustering did. That throws away something real: `a ship's fuel consumption this month is not independent of its consumption in recent months` — routes, maintenance cycles, and seasonal weather patterns carry over. **Recurrent Neural Networks (RNNs)** are built specifically to carry information forward from one step in a sequence to the next, instead of treating every row as a fresh, unrelated draw.

---

## 3. The RNN recurrence relation, unrolled through time

An **RNN cell** takes the current input $x_t$ *and* its own previous output — the **hidden state** $h_{t-1}$ — and produces a new hidden state:

$$
h_t = \tanh(W_x x_t + W_h h_{t-1} + b)
$$

$W_x$, $W_h$, and $b$ are the **same weights at every time step** — one small cell, applied repeatedly, carrying $h_t$ forward as memory of everything seen so far. "Unrolling" draws that one cell once per time step, to see the whole sequence at a glance:

Let's draw that unrolled structure for a 4-step sequence:

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(11, 4.5))

n_steps = 4
box_w, box_h = 1.2, 1.0
xs = [i * 2.6 for i in range(n_steps)]

for i, x in enumerate(xs):
    ax.add_patch(patches.FancyBboxPatch((x, 0), box_w, box_h, boxstyle="round,pad=0.05",
                                         facecolor="lightcoral", edgecolor="black"))
    ax.text(x + box_w / 2, box_h / 2, "RNN\ncell", ha="center", va="center", fontsize=9)

    ax.annotate("", xy=(x + box_w / 2, 0), xytext=(x + box_w / 2, -1),
                arrowprops=dict(arrowstyle="->", lw=1.5))
    ax.text(x + box_w / 2, -1.3, f"$x_{i + 1}$", ha="center", fontsize=11)

    ax.annotate("", xy=(x + box_w / 2, box_h + 1), xytext=(x + box_w / 2, box_h),
                arrowprops=dict(arrowstyle="->", lw=1.5))
    ax.text(x + box_w / 2, box_h + 1.3, f"$y_{i + 1}$", ha="center", fontsize=11)

    if i < n_steps - 1:
        ax.annotate("", xy=(xs[i + 1], box_h / 2), xytext=(x + box_w, box_h / 2),
                    arrowprops=dict(arrowstyle="->", lw=2, color="darkblue"))
        ax.text(x + box_w + (xs[i + 1] - x - box_w) / 2, box_h / 2 + 0.3,
                f"$h_{i + 1}$", ha="center", color="darkblue", fontsize=10)

ax.annotate("", xy=(xs[0], box_h / 2), xytext=(xs[0] - 0.8, box_h / 2),
            arrowprops=dict(arrowstyle="->", lw=2, color="darkblue"))
ax.text(xs[0] - 1.0, box_h / 2 + 0.3, "$h_0$", ha="center", color="darkblue", fontsize=10)

ax.text(xs[-1] / 2, -2.0, "Same cell, same weights, reused at every time step",
        ha="center", fontsize=10, style="italic")

ax.set_xlim(-2, xs[-1] + box_w + 1)
ax.set_ylim(-2.5, box_h + 2)
ax.axis("off")
ax.set_title("An RNN unrolled through time")
plt.tight_layout()
plt.show()

This is one RNN cell drawn four times, not four different cells — the blue arrows carrying $h_t$ forward are how `information from early in a sequence (say, January's fuel efficiency) can influence a prediction much later in it` (say, a forecast for December).

> **Further reading**: [Recurrent neural network (Wikipedia)](https://en.wikipedia.org/wiki/Recurrent_neural_network).

---

## 4. Worked example: computing an RNN's hidden states by hand

Small, untrained, illustrative numbers — the same spirit as `NB13`'s hand-computed convolution — so the recurrence formula stops being abstract:

In [ ]:
import numpy as np

def rnn_cell(x, h_prev, Wx, Wh, b):
    return np.tanh(Wx * x + Wh * h_prev + b)

# Toy weights -- not learned, just for illustration
Wx, Wh, b = 0.5, 0.8, 0.1

x_sequence = [1.0, 0.5, -0.5, 1.0]
h = 0.0  # initial hidden state h0

print(f"h0 = {h:.3f}")
for t, x_t in enumerate(x_sequence, start=1):
    h = rnn_cell(x_t, h, Wx, Wh, b)
    print(f"x{t} = {x_t:>5} -> h{t} = {h:.3f}")

**Trace it yourself**: each $h_t$ depends on both the current $x_t$ *and* every $h$ before it, through the chain of substitutions — $h_4$ is influenced by $x_1$, three steps earlier, entirely through $h_1 \to h_2 \to h_3$.

**Try it yourself**: change the toy weights and feed in a single "pulse" — one non-zero input at the start, then zeros — to see the memory effect directly: how much does that first input still influence $h_4$, three steps later, purely through $W_h$?

In [ ]:
Wx2, Wh2, b2 = 0.2, 1.2, -0.1
x_sequence2 = [1.0, 0.0, 0.0, 0.0]   # a single pulse at t=1, then nothing

h2 = 0.0
print(f"h0 = {h2:.3f}")
for t, x_t in enumerate(x_sequence2, start=1):
    h2 = rnn_cell(x_t, h2, Wx2, Wh2, b2)
    print(f"x{t} = {x_t:>5} -> h{t} = {h2:.3f}")


---

## 5. The vanishing gradient problem

Training an RNN uses **backpropagation through time** — the chain rule stretched back through every time step. Since the *same* weight $W_h$ multiplies the hidden state at every step, the gradient flowing back through $n$ steps involves that weight (and $\tanh$'s derivative, always $\le 1$) multiplied by itself roughly $n$ times. If that per-step factor is even slightly below 1, repeated multiplication shrinks it **exponentially** — illustrated below with plain arithmetic, not a real trained model:

In [ ]:
factor = 0.6  # a representative per-step gradient factor below 1
steps = np.arange(0, 13)
magnitude = factor ** steps

plt.figure(figsize=(7, 4))
plt.plot(steps, magnitude, marker="o")
plt.yscale("log")
plt.xlabel("Number of time steps back-propagated through")
plt.ylabel("Relative gradient magnitude (log scale)")
plt.title(f"Illustrative effect of repeated multiplication by {factor} per step")
plt.show()

By 10–12 steps back, this illustrative gradient is already orders of magnitude smaller than at step 1 — `for a plain RNN, information from that far back barely influences learning`. Our sequences today are only 11 steps long, right in the range where this starts to bite — a good practical reason to reach for LSTM instead of a plain RNN.

> **Further reading**: [Vanishing gradient problem (Wikipedia)](https://en.wikipedia.org/wiki/Vanishing_gradient_problem).

---

## 6. LSTM: gates and cell state

**Long Short-Term Memory (LSTM)** adds a second pathway — the **cell state** $C_t$ — carried across time steps mostly by *addition* rather than repeated multiplication, controlled by three learned gates:

| Gate | Question it answers |
|---|---|
| **Forget gate** $f_t$ | How much of the old cell state should we keep? |
| **Input gate** $i_t$ | How much new information should we add? |
| **Output gate** $o_t$ | How much of the cell state should influence this step's output? |

$$
C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t \qquad h_t = o_t \odot \tanh(C_t)
$$

Because $C_t$'s update is dominated by addition, not a repeated multiplicative shrinkage, `LSTMs handle much longer dependencies than plain RNNs before the vanishing-gradient problem from Part 5 takes over`. The simpler **[GRU](https://en.wikipedia.org/wiki/Gated_recurrent_unit)** (Gated Recurrent Unit) merges the cell and hidden states and uses only two gates — fewer parameters, often competitive on shorter sequences like ours; swapping `nn.LSTM` for `nn.GRU` later is a one-line change, worth trying as homework.

> **Further reading**: [Long short-term memory (Wikipedia)](https://en.wikipedia.org/wiki/Long_short-term_memory).

---

## 7. Real dataset: ship fuel data as a proper time series

Reload the same real `ship_fuel_efficiency.csv` from `NB07`/`NB09` — 120 ships, each with 12 chronological monthly records:

In [ ]:
!wget -q -O ship_fuel_efficiency.csv https://raw.githubusercontent.com/JuanZapa7a/AINavalEngineering/main/Datasets/ship_fuel_efficiency.csv

import pandas as pd

fuel = pd.read_csv("ship_fuel_efficiency.csv")
print(fuel.shape)
fuel[fuel["ship_id"] == fuel["ship_id"].iloc[0]][["ship_id", "month", "distance", "fuel_consumption", "engine_efficiency"]]

**Today's task**: using each ship's first **11 months** (`distance`, `fuel_consumption`, `engine_efficiency` — the same leakage-safe feature set from `NB07`, still excluding `CO2_emissions`) as an input sequence, predict that ship's **12th-month `fuel_consumption`**. Build one (11-step, 3-feature) sequence per ship, plus its target:

In [ ]:
import numpy as np

feature_cols = ["distance", "fuel_consumption", "engine_efficiency"]
ship_ids = fuel["ship_id"].unique()

sequences, targets = [], []
for sid in ship_ids:
    ship_rows = fuel[fuel["ship_id"] == sid][feature_cols].values  # already chronological, Jan-Dec
    sequences.append(ship_rows[:11])          # months 1-11 as input
    targets.append(ship_rows[11, 1])          # month 12's fuel_consumption (column index 1)

X_seq = np.stack(sequences)   # (120 ships, 11 months, 3 features)
y_seq = np.array(targets)     # (120,)
print(X_seq.shape, y_seq.shape)

Split **by ship**, not by row — putting different months of the *same* ship into both train and test would leak information about that ship's overall fuel behavior across the split, echoing `NB07`'s leakage lesson in a new, sequence-specific form:

In [ ]:
from sklearn.model_selection import train_test_split

idx = np.arange(len(ship_ids))
idx_train_full, idx_test = train_test_split(idx, test_size=0.2, random_state=42)
idx_train, idx_val = train_test_split(idx_train_full, test_size=0.25, random_state=42)

print(f"Train ships: {len(idx_train)}  Val ships: {len(idx_val)}  Test ships: {len(idx_test)}")

**Try it yourself**: confirm the split-by-ship promise directly — check that no `ship_id` appears in more than one of the three splits.

In [ ]:
train_ships = set(ship_ids[idx_train])
val_ships = set(ship_ids[idx_val])
test_ships = set(ship_ids[idx_test])

print("Train/val overlap:", train_ships & val_ships)
print("Train/test overlap:", train_ships & test_ships)
print("Val/test overlap:", val_ships & test_ships)
print("All three splits together cover every ship?", len(train_ships | val_ships | test_ships) == len(ship_ids))


---

## 8. Hands-on: building sequences and training an LSTM forecaster

Scale features using statistics from the **training sequences only** (fit on the flattened train data, exactly like every earlier notebook's leakage-safe scaling), then convert to tensors:

In [ ]:
from sklearn.preprocessing import StandardScaler
import torch

scaler = StandardScaler()
scaler.fit(X_seq[idx_train].reshape(-1, X_seq.shape[-1]))  # flatten (ships, months) to fit

def scale_sequences(X):
    shape = X.shape
    return scaler.transform(X.reshape(-1, shape[-1])).reshape(shape)

X_train_t = torch.tensor(scale_sequences(X_seq[idx_train]), dtype=torch.float32)
X_val_t = torch.tensor(scale_sequences(X_seq[idx_val]), dtype=torch.float32)
X_test_t = torch.tensor(scale_sequences(X_seq[idx_test]), dtype=torch.float32)

y_train_t = torch.tensor(y_seq[idx_train], dtype=torch.float32).view(-1, 1)
y_val_t = torch.tensor(y_seq[idx_val], dtype=torch.float32).view(-1, 1)
y_test_t = torch.tensor(y_seq[idx_test], dtype=torch.float32).view(-1, 1)

X_train_t.shape

Define a small LSTM forecaster: process the 11-month sequence, take the **final** hidden state (a summary of the whole sequence — exactly what `NB11` §9 previewed), and map it to one predicted value:

In [ ]:
import torch.nn as nn

class FuelLSTM(nn.Module):
    def __init__(self, n_features, hidden_size=16):
        super().__init__()
        self.lstm = nn.LSTM(input_size=n_features, hidden_size=hidden_size, batch_first=True)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        return self.head(h_n[-1])

torch.manual_seed(42)
lstm_model = FuelLSTM(n_features=X_train_t.shape[-1])
sum(p.numel() for p in lstm_model.parameters())

Train with the same recipe as every network since `NB11`: Adam, tracking train and validation loss. Since we're predicting a continuous value (fuel consumption, not a class), we use **MSE loss** — `NB07`'s regression metric — instead of the binary cross-entropy from `NB11`/`NB12`/`NB13`:

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(lstm_model.parameters(), lr=0.01)

n_epochs = 150
train_losses, val_losses = [], []

for epoch in range(n_epochs):
    lstm_model.train()
    optimizer.zero_grad()
    outputs = lstm_model(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward()
    optimizer.step()
    train_losses.append(loss.item())

    lstm_model.eval()
    with torch.no_grad():
        val_loss = criterion(lstm_model(X_val_t), y_val_t)
    val_losses.append(val_loss.item())

plt.plot(train_losses, label="Training loss (MSE)")
plt.plot(val_losses, label="Validation loss (MSE)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("LSTM training: forecasting month-12 fuel consumption")
plt.legend()
plt.show()

**Try it yourself**: Section 5 argued LSTM should handle longer dependencies better than a plain RNN. Test that claim directly — swap `nn.LSTM` for `nn.RNN` (same constructor pattern, but its `forward` returns just `(output, h_n)`, no separate cell state) and train an equivalent model on the exact same data. How different are the final validation losses?

In [ ]:
class FuelRNN(nn.Module):
    def __init__(self, n_features, hidden_size=16):
        super().__init__()
        self.rnn = nn.RNN(input_size=n_features, hidden_size=hidden_size, batch_first=True)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        _, h_n = self.rnn(x)
        return self.head(h_n[-1])

torch.manual_seed(42)
rnn_model = FuelRNN(n_features=X_train_t.shape[-1])
optimizer_rnn = torch.optim.Adam(rnn_model.parameters(), lr=0.01)

rnn_val_losses = []
for epoch in range(n_epochs):
    rnn_model.train()
    optimizer_rnn.zero_grad()
    loss = criterion(rnn_model(X_train_t), y_train_t)
    loss.backward()
    optimizer_rnn.step()

    rnn_model.eval()
    with torch.no_grad():
        rnn_val_losses.append(criterion(rnn_model(X_val_t), y_val_t).item())

print(f"LSTM final validation loss: {val_losses[-1]:.1f}")
print(f"Plain RNN final validation loss: {rnn_val_losses[-1]:.1f}")


On sequences this short (11 steps) and this little data, plain RNN and LSTM often land close together — the vanishing-gradient advantage LSTM is built for has more room to show up on longer sequences or larger datasets. That doesn't make Section 5's theory wrong; it just means this particular practical comparison may not be where the gap is most visible.

---

## 9. Evaluating the LSTM against an honest baseline

A model is only impressive relative to something simpler. The obvious baseline for a time series: **predict that next month equals last month** (month 12 ≈ month 11's `fuel_consumption`) — no learning at all, just persistence. If the LSTM can't beat this, it isn't earning its complexity:

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

# Naive baseline: predict month 12 = month 11's fuel_consumption
baseline_pred = X_seq[idx_test][:, -1, 1]  # last month in the input sequence, fuel_consumption column
baseline_mae = mean_absolute_error(y_seq[idx_test], baseline_pred)

# LSTM prediction
lstm_model.eval()
with torch.no_grad():
    lstm_pred = lstm_model(X_test_t).numpy().ravel()
lstm_mae = mean_absolute_error(y_seq[idx_test], lstm_pred)
lstm_r2 = r2_score(y_seq[idx_test], lstm_pred)

print(f"Naive baseline MAE: {baseline_mae:.1f} L")
print(f"LSTM MAE:           {lstm_mae:.1f} L")
print(f"LSTM R2:             {lstm_r2:.3f}")

**Read your own numbers**: does the LSTM beat the naive baseline? With only 72 training sequences (one per training ship) and 11 time steps each, this is a genuinely small dataset for a neural network — it's entirely possible the naive baseline wins here, and `that would be a legitimate, honest result, not a failure of the notebook`. Recall `NB08` §9's broader lesson: neural networks need more data than tree-based methods to show a clear advantage, and 72 sequences is small even by this course's standards. A visual check helps interpret whichever way it goes:

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_seq[idx_test], lstm_pred, alpha=0.7, label="LSTM")
plt.scatter(y_seq[idx_test], baseline_pred, alpha=0.7, label="Naive baseline", marker="x")
lims = [y_seq.min(), y_seq.max()]
plt.plot(lims, lims, "r--", label="Perfect prediction")
plt.xlabel("Actual month-12 fuel consumption (L)")
plt.ylabel("Predicted (L)")
plt.title("LSTM vs. naive baseline, test ships")
plt.legend()
plt.show()

**Try it yourself**: break the LSTM's absolute error down by `ship_type` — is the model's forecast equally reliable across all four vessel types, or does it struggle more on one?

In [ ]:
test_ship_ids = ship_ids[idx_test]
ship_types = fuel.drop_duplicates("ship_id").set_index("ship_id")["ship_type"]

errors_df = pd.DataFrame({
    "ship_type": [ship_types[s] for s in test_ship_ids],
    "abs_error": np.abs(y_seq[idx_test] - lstm_pred),
})
errors_df.groupby("ship_type")["abs_error"].agg(["mean", "count"])


---

## Class summary

- Sequence data carries meaning in its order; an RNN cell reuses the same weights at every time step, carrying a hidden state forward as memory.
- We computed a small RNN's hidden states by hand, and saw why repeated multiplication through many time steps causes gradients to vanish.
- LSTM's gated cell state, updated mostly by addition, resists this far better than a plain RNN.
- We reframed real ship fuel data — already used twice in this course as row-independent tabular data — as what it actually is: a 12-month time series per ship, split *by ship* to avoid leakage.
- We trained a real LSTM forecaster and judged it against an honest naive baseline, not just against its own training loss.

## For the next class (NB15)

**Transfer learning**: instead of training a CNN from scratch as in `NB13`, we'll reuse a network already trained on millions of images — directly following up on `NB13`'s closing point about why early convolutional filters generalize across tasks.

## Homework / Practice Ideas

1. Change `hidden_size` from 16 to 32 and to 4 — how does test MAE change? Relate this to the parameter-count/overfitting discussion from `NB11`–`NB13`.
2. Swap `nn.LSTM` for `nn.GRU` in Part 8 (same constructor arguments work, though `nn.GRU` returns only `(output, h_n)`, no separate cell state) — does it perform differently with this little data?
3. Add `route_id` or `weather_conditions` (one-hot encoded) as extra sequence features — does the LSTM improve, and is the added complexity worth it given how small this dataset is?
4. Try predicting month 6 from months 1–5 instead of month 12 from months 1–11 — does a shorter sequence make the vanishing-gradient concern from Part 5 less relevant in practice?
5. In your own words, explain why splitting by *ship* (Part 7) rather than by *row* was necessary here — what exactly would leak if we split randomly by row instead?

> ***As always: a model that loses to a one-line baseline is a real, useful result — it tells you the data (or the amount of it) doesn't support the complexity you tried, which is worth knowing before deploying anything.***
